In [1]:
# Cell 1 — Imports
from dotenv import load_dotenv
import os
load_dotenv(r'C:\Users\Gurveer\ds-portfolio\.env')
os.environ["OPENAI_API_KEY"] = os.getenv("OPENAI_API_KEY")
SERPAPI_KEY = os.getenv("SERPAPI_KEY")

import numpy as np
import pandas as pd
import plotly.graph_objects as go
import plotly.express as px
import math
import json
import requests
from openai import OpenAI
import warnings
warnings.filterwarnings('ignore')

client = OpenAI()
print("All imports successful")
print(f"SerpAPI key loaded: {'Yes' if SERPAPI_KEY else 'NO — check .env'}")

All imports successful
SerpAPI key loaded: Yes


In [2]:
# Cell 2 — IFR Core Mathematics Engine
# All calculations from first principles — no external aviation libraries

def haversine(lat1, lon1, lat2, lon2):
    """Great circle distance in nautical miles"""
    R = 3440.065  # Earth radius in NM
    lat1, lon1, lat2, lon2 = map(math.radians, [lat1, lon1, lat2, lon2])
    dlat = lat2 - lat1
    dlon = lon2 - lon1
    a = math.sin(dlat/2)**2 + math.cos(lat1)*math.cos(lat2)*math.sin(dlon/2)**2
    return R * 2 * math.asin(math.sqrt(a))

def initial_bearing(lat1, lon1, lat2, lon2):
    """True course from point 1 to point 2"""
    lat1, lon1, lat2, lon2 = map(math.radians, [lat1, lon1, lat2, lon2])
    dlon = lon2 - lon1
    x = math.sin(dlon) * math.cos(lat2)
    y = math.cos(lat1)*math.sin(lat2) - math.sin(lat1)*math.cos(lat2)*math.cos(dlon)
    return (math.degrees(math.atan2(x, y)) + 360) % 360

def magnetic_variation(lat, lon):
    """Approximate magnetic variation (simplified model)"""
    return -14.0  # Boston area approx

def e6b_wind_correction(tas_kts, wind_dir, wind_spd, true_course):
    """
    E6B wind correction angle and ground speed
    Returns: (wca_deg, ground_speed_kts, magnetic_heading)
    """
    tc_rad  = math.radians(true_course)
    wd_rad  = math.radians(wind_dir)
    
    # Wind components
    hwc = wind_spd * math.cos(wd_rad - tc_rad)  # headwind component
    xwc = wind_spd * math.sin(wd_rad - tc_rad)  # crosswind component
    
    # Wind correction angle
    wca = math.degrees(math.asin(xwc / tas_kts)) if abs(xwc/tas_kts) <= 1 else 0
    
    # Ground speed
    gs = tas_kts * math.cos(math.radians(wca)) - hwc
    
    # Magnetic heading
    mh = (true_course + wca + magnetic_variation(0, 0) + 360) % 360
    
    return round(wca, 1), round(gs, 1), round(mh, 1)

def density_altitude(elevation_ft, temp_c, altimeter_inhg=29.92):
    """Calculate density altitude in feet"""
    pressure_alt = elevation_ft + (29.92 - altimeter_inhg) * 1000
    isa_temp_c   = 15 - (0.00198 * elevation_ft)
    da            = pressure_alt + 118.8 * (temp_c - isa_temp_c)
    return round(da, 0)

def tas_from_cas(cas_kts, altitude_ft, temp_c=None):
    """Convert CAS to TAS"""
    if temp_c is None:
        temp_c = 15 - 0.00198 * altitude_ft
    pressure_ratio = (1 - altitude_ft / 145442) ** 5.2561
    temp_ratio     = (temp_c + 273.15) / 288.15
    eas = cas_kts * math.sqrt(pressure_ratio)
    tas = eas / math.sqrt(temp_ratio * pressure_ratio)
    return round(tas, 1)

def fuel_time_distance(gs_kts, distance_nm, fuel_burn_gph):
    """Calculate time en route and fuel required"""
    time_hr   = distance_nm / gs_kts
    time_min  = time_hr * 60
    fuel_gal  = time_hr * fuel_burn_gph
    return round(time_min, 1), round(fuel_gal, 1)

# Test all functions
print("=== E6B Core Mathematics Test ===")
wca, gs, mh = e6b_wind_correction(
    tas_kts=120, wind_dir=270, wind_spd=20, true_course=360
)
print(f"Wind Correction Angle: {wca}°")
print(f"Ground Speed:          {gs} kts")
print(f"Magnetic Heading:      {mh}°")

da = density_altitude(elevation_ft=1000, temp_c=30)
print(f"\nDensity Altitude:      {da} ft")

tas = tas_from_cas(cas_kts=110, altitude_ft=8000)
print(f"TAS from 110 KCAS @ 8000ft: {tas} kts")

time_min, fuel = fuel_time_distance(gs_kts=115, distance_nm=50, fuel_burn_gph=8.5)
print(f"\n50 NM @ 115 kts GS:")
print(f"  Time:  {time_min} min")
print(f"  Fuel:  {fuel} gal")

=== E6B Core Mathematics Test ===
Wind Correction Angle: -9.6°
Ground Speed:          118.3 kts
Magnetic Heading:      336.4°

Density Altitude:      3017.0 ft
TAS from 110 KCAS @ 8000ft: 113.2 kts

50 NM @ 115 kts GS:
  Time:  26.1 min
  Fuel:  3.7 gal


In [4]:
# Cell 3 — Holding Pattern Calculator
def holding_pattern_entry(inbound_course, aircraft_heading):
    """
    Determine IFR holding pattern entry type per FAA/ICAO standards
    Returns: entry type and procedure description
    """
    # Relative bearing from inbound course
    rel = (aircraft_heading - inbound_course + 360) % 360
    
    if 0 <= rel <= 70 or 290 <= rel <= 360:
        entry = "Direct"
        desc  = (f"Fly direct to the fix, turn RIGHT to the outbound heading "
                 f"({(inbound_course + 180) % 360}°), complete the pattern.")
    elif 71 <= rel <= 180:
        entry = "Teardrop"
        desc  = (f"Cross the fix, fly outbound on a heading of "
                 f"{(inbound_course + 150) % 360}° for 1 minute, "
                 f"then turn RIGHT to intercept the inbound course {inbound_course}°.")
    else:
        entry = "Parallel"
        desc  = (f"Cross the fix, turn LEFT to parallel the outbound course "
                 f"({(inbound_course + 180) % 360}°) for 1 minute, "
                 f"then turn RIGHT to intercept inbound course {inbound_course}°.")
    
    return entry, desc

def holding_pattern_timing(altitude_ft, wind_spd_kts=0):
    """
    Calculate holding pattern leg timing per FAA standards
    Below 14000: 1 min legs, Above: 1.5 min legs
    Wind correction for outbound leg
    """
    if altitude_ft <= 14000:
        inbound_time  = 1.0
        outbound_time = 1.0 + (wind_spd_kts * 0.01)
    else:
        inbound_time  = 1.5
        outbound_time = 1.5 + (wind_spd_kts * 0.015)
    
    return round(inbound_time, 2), round(outbound_time, 2)

def holding_pattern_size(tas_kts, bank_angle=25):
    """Calculate holding pattern dimensions"""
    turn_radius_nm = tas_kts / (3600 * math.tan(math.radians(bank_angle)) * 
                                math.pi / (tas_kts * 0.01745))
    turn_radius_nm = (tas_kts ** 2) / (11.3 * math.tan(math.radians(bank_angle)) * 3600)
    pattern_width  = 2 * turn_radius_nm
    return round(turn_radius_nm, 2), round(pattern_width, 2)

# Test holding pattern calculator
print("=== HOLDING PATTERN CALCULATOR ===\n")

test_cases = [
    (180, 45,  "Approaching from NE"),
    (180, 200, "Approaching from S"),
    (180, 300, "Approaching from NW"),
    (90,  135, "Approaching from SE"),
]

for inbound, heading, scenario in test_cases:
    entry, desc = holding_pattern_entry(inbound, heading)
    in_time, out_time = holding_pattern_timing(8000, wind_spd_kts=15)
    print(f"Scenario: {scenario}")
    print(f"  Inbound course:  {inbound}°")
    print(f"  Aircraft heading:{heading}°")
    print(f"  Entry type:      {entry}")
    print(f"  Procedure:       {desc}")
    print(f"  Timing:          {in_time} min inbound / {out_time} min outbound")
    print()

=== HOLDING PATTERN CALCULATOR ===

Scenario: Approaching from NE
  Inbound course:  180°
  Aircraft heading:45°
  Entry type:      Parallel
  Procedure:       Cross the fix, turn LEFT to parallel the outbound course (0°) for 1 minute, then turn RIGHT to intercept inbound course 180°.
  Timing:          1.0 min inbound / 1.15 min outbound

Scenario: Approaching from S
  Inbound course:  180°
  Aircraft heading:200°
  Entry type:      Direct
  Procedure:       Fly direct to the fix, turn RIGHT to the outbound heading (0°), complete the pattern.
  Timing:          1.0 min inbound / 1.15 min outbound

Scenario: Approaching from NW
  Inbound course:  180°
  Aircraft heading:300°
  Entry type:      Teardrop
  Procedure:       Cross the fix, fly outbound on a heading of 330° for 1 minute, then turn RIGHT to intercept the inbound course 180°.
  Timing:          1.0 min inbound / 1.15 min outbound

Scenario: Approaching from SE
  Inbound course:  90°
  Aircraft heading:135°
  Entry type:      

In [5]:
# Cell 4 — Flight Plan Generator
# Major airports database
airports = {
    'KBOS': {'name': 'Boston Logan',      'lat': 42.3656, 'lon': -71.0096, 'elev': 19,   'freq': '119.1'},
    'KJFK': {'name': 'JFK International', 'lat': 40.6413, 'lon': -73.7781, 'elev': 13,   'freq': '119.1'},
    'KORH': {'name': 'Worcester Regional','lat': 42.2673, 'lon': -71.8757, 'elev': 1009, 'freq': '124.0'},
    'KBDL': {'name': 'Bradley Intl',      'lat': 41.9389, 'lon': -72.6832, 'elev': 173,  'freq': '124.0'},
    'KMHT': {'name': 'Manchester-Boston', 'lat': 42.9326, 'lon': -71.4357, 'elev': 235,  'freq': '121.3'},
    'KPVD': {'name': 'Providence TF Green','lat':41.7272, 'lon': -71.4284, 'elev': 55,   'freq': '119.4'},
    'KLGA': {'name': 'LaGuardia',         'lat': 40.7772, 'lon': -73.8726, 'elev': 21,   'freq': '119.9'},
    'KORD': {'name': 'Chicago O\'Hare',   'lat': 41.9742, 'lon': -87.9073, 'elev': 672,  'freq': '135.4'},
    'KATL': {'name': 'Atlanta Hartsfield','lat': 33.6407, 'lon': -84.4277, 'elev': 1026, 'freq': '119.5'},
    'KMIA': {'name': 'Miami Intl',        'lat': 25.7959, 'lon': -80.2870, 'elev': 8,    'freq': '119.7'},
}

def generate_flight_plan(dep, dest, cruise_alt_ft, tas_kts,
                          fuel_burn_gph, wind_dir, wind_spd,
                          temp_c=-20, altimeter=29.92):
    """Generate a complete IFR flight plan"""
    if dep not in airports or dest not in airports:
        return None
    
    dep_info  = airports[dep]
    dest_info = airports[dest]
    
    # Navigation calculations
    dist_nm   = haversine(dep_info['lat'], dep_info['lon'],
                           dest_info['lat'], dest_info['lon'])
    true_crs  = initial_bearing(dep_info['lat'], dep_info['lon'],
                                 dest_info['lat'], dest_info['lon'])
    wca, gs, mh = e6b_wind_correction(tas_kts, wind_dir, wind_spd, true_crs)
    time_min, fuel_gal = fuel_time_distance(gs, dist_nm, fuel_burn_gph)
    da = density_altitude(dep_info['elev'], temp_c, altimeter)
    actual_tas = tas_from_cas(tas_kts, cruise_alt_ft, temp_c)
    
    # Alternate fuel (45 min reserve for IFR)
    reserve_fuel = fuel_burn_gph * 0.75
    total_fuel   = fuel_gal + reserve_fuel
    
    plan = {
        'departure':        f"{dep} — {dep_info['name']}",
        'destination':      f"{dest} — {dest_info['name']}",
        'cruise_altitude':  f"{cruise_alt_ft:,} ft MSL",
        'distance':         f"{dist_nm:.1f} NM",
        'true_course':      f"{true_crs:.1f}°",
        'magnetic_heading': f"{mh:.1f}°",
        'wind':             f"{wind_dir}° @ {wind_spd} kts",
        'wind_correction':  f"{wca:+.1f}°",
        'tas':              f"{actual_tas} kts",
        'ground_speed':     f"{gs} kts",
        'est_time':         f"{int(time_min//60)}h {int(time_min%60)}m",
        'fuel_enroute':     f"{fuel_gal:.1f} gal",
        'fuel_reserve':     f"{reserve_fuel:.1f} gal (45 min IFR)",
        'fuel_total':       f"{total_fuel:.1f} gal",
        'density_altitude': f"{da:,} ft",
    }
    return plan

def print_flight_plan(plan):
    """Print formatted flight plan table"""
    if not plan:
        print("Invalid airport codes")
        return
    print(f"\n{'='*55}")
    print(f"  IFR FLIGHT PLAN")
    print(f"{'='*55}")
    for key, val in plan.items():
        print(f"  {key.replace('_',' ').title():<22} {val}")
    print(f"{'='*55}")

# Generate sample flight plans
print("FLIGHT PLAN 1: KBOS → KJFK")
plan1 = generate_flight_plan(
    dep='KBOS', dest='KJFK',
    cruise_alt_ft=8000, tas_kts=120,
    fuel_burn_gph=8.5, wind_dir=270,
    wind_spd=25, temp_c=-10
)
print_flight_plan(plan1)

print("\nFLIGHT PLAN 2: KBOS → KORD")
plan2 = generate_flight_plan(
    dep='KBOS', dest='KORD',
    cruise_alt_ft=10000, tas_kts=130,
    fuel_burn_gph=9.0, wind_dir=310,
    wind_spd=30, temp_c=-15
)
print_flight_plan(plan2)

FLIGHT PLAN 1: KBOS → KJFK

  IFR FLIGHT PLAN
  Departure              KBOS — Boston Logan
  Destination            KJFK — JFK International
  Cruise Altitude        8,000 ft MSL
  Distance               161.9 NM
  True Course            231.2°
  Magnetic Heading       224.7°
  Wind                   270° @ 25 kts
  Wind Correction        +7.5°
  Tas                    125.6 kts
  Ground Speed           99.5 kts
  Est Time               1h 37m
  Fuel Enroute           13.8 gal
  Fuel Reserve           6.4 gal (45 min IFR)
  Fuel Total             20.2 gal
  Density Altitude       -2,947.0 ft

FLIGHT PLAN 2: KBOS → KORD

  IFR FLIGHT PLAN
  Departure              KBOS — Boston Logan
  Destination            KORD — Chicago O'Hare
  Cruise Altitude        10,000 ft MSL
  Distance               751.1 NM
  True Course            273.9°
  Magnetic Heading       267.7°
  Wind                   310° @ 30 kts
  Wind Correction        +7.8°
  Tas                    137.3 kts
  Ground Speed      

In [6]:
# Cell 5 — PilotGPT AI Assistant (OpenAI powered)
from langchain_openai import ChatOpenAI
from langchain_core.tools import tool
from langgraph.prebuilt import create_react_agent
import json

llm = ChatOpenAI(model="gpt-4o-mini", temperature=0.2)

@tool
def calculate_holding_entry(inbound_course: int, aircraft_heading: int) -> str:
    """Calculate IFR holding pattern entry type given inbound course and aircraft heading."""
    entry, desc = holding_pattern_entry(inbound_course, aircraft_heading)
    in_time, out_time = holding_pattern_timing(8000)
    return json.dumps({
        "entry_type": entry,
        "procedure": desc,
        "inbound_time_min": in_time,
        "outbound_time_min": out_time
    })

@tool
def calculate_wind_correction(tas_kts: float, wind_dir: int, 
                               wind_spd: int, true_course: int) -> str:
    """Calculate wind correction angle, ground speed, and magnetic heading."""
    wca, gs, mh = e6b_wind_correction(tas_kts, wind_dir, wind_spd, true_course)
    return json.dumps({
        "wind_correction_angle_deg": wca,
        "ground_speed_kts": gs,
        "magnetic_heading_deg": mh
    })

@tool
def calculate_density_altitude(elevation_ft: int, temp_c: float, 
                                altimeter_inhg: float = 29.92) -> str:
    """Calculate density altitude from field elevation, temperature, and altimeter setting."""
    da = density_altitude(elevation_ft, temp_c, altimeter_inhg)
    isa_temp = 15 - (0.00198 * elevation_ft)
    return json.dumps({
        "density_altitude_ft": da,
        "pressure_altitude_ft": elevation_ft + (29.92 - altimeter_inhg) * 1000,
        "isa_temp_c": round(isa_temp, 1),
        "temp_deviation_c": round(temp_c - isa_temp, 1)
    })

@tool
def generate_ifr_flight_plan(departure: str, destination: str,
                              cruise_alt_ft: int, tas_kts: float,
                              fuel_burn_gph: float, wind_dir: int,
                              wind_spd: int) -> str:
    """Generate a complete IFR flight plan between two airports."""
    plan = generate_flight_plan(
        departure, destination, cruise_alt_ft,
        tas_kts, fuel_burn_gph, wind_dir, wind_spd
    )
    return json.dumps(plan) if plan else "Invalid airport codes"

@tool
def get_airport_info(icao_code: str) -> str:
    """Get airport information by ICAO code."""
    if icao_code in airports:
        info = airports[icao_code]
        return json.dumps({
            "icao": icao_code,
            "name": info['name'],
            "elevation_ft": info['elev'],
            "atis_freq": info['freq']
        })
    return f"Airport {icao_code} not in database"

tools = [calculate_holding_entry, calculate_wind_correction,
         calculate_density_altitude, generate_ifr_flight_plan,
         get_airport_info]

pilotgpt = create_react_agent(llm, tools=tools)
print(f"PilotGPT agent ready with {len(tools)} tools")
print(f"Tools: {[t.name for t in tools]}")

PilotGPT agent ready with 5 tools
Tools: ['calculate_holding_entry', 'calculate_wind_correction', 'calculate_density_altitude', 'generate_ifr_flight_plan', 'get_airport_info']


In [7]:
# Cell 6 — Run PilotGPT Queries
queries = [
    "I'm flying inbound on the 180 radial to a VOR holding pattern. My current heading is 300 degrees. What entry should I use and what's the procedure?",
    "I'm planning a flight from KBOS to KJFK at 8000 feet. TAS is 120 knots, winds are 270 at 25 knots. What's my wind correction angle, ground speed, and magnetic heading?",
    "What is the density altitude at KORH (Worcester) with an OAT of 35°C and altimeter 29.75?",
    "Generate a complete IFR flight plan from KBOS to KATL at 10000 feet, TAS 130 knots, fuel burn 9 GPH, winds 300 at 20 knots.",
]

print("="*60)
print("  PILOTGPT v2 — IFR ASSISTANT")
print("="*60)

for q in queries:
    print(f"\nPilot: {q}")
    print("-"*60)
    result = pilotgpt.invoke({"messages": [{"role": "user", "content": q}]})
    print(f"PilotGPT: {result['messages'][-1].content}")
    print()

  PILOTGPT v2 — IFR ASSISTANT

Pilot: I'm flying inbound on the 180 radial to a VOR holding pattern. My current heading is 300 degrees. What entry should I use and what's the procedure?
------------------------------------------------------------
PilotGPT: For your inbound flight on the 180 radial with a current heading of 300 degrees, you should use a **Teardrop entry**.

### Procedure:
1. **Cross the fix** (the VOR).
2. Fly outbound on a heading of **330°** for **1 minute**.
3. After 1 minute, turn **RIGHT** to intercept the inbound course of **180°**.

This entry allows you to establish yourself in the holding pattern effectively.


Pilot: I'm planning a flight from KBOS to KJFK at 8000 feet. TAS is 120 knots, winds are 270 at 25 knots. What's my wind correction angle, ground speed, and magnetic heading?
------------------------------------------------------------
PilotGPT: For your flight from KBOS to KJFK at 8000 feet with a true airspeed of 120 knots and winds from 270° at 25 kno

In [8]:
# Cell 7 — Export Flight Plans & Summary
import os

output_dir = r'C:\Users\Gurveer\ds-portfolio\project-26-pilotgpt-v2\outputs'
os.makedirs(output_dir, exist_ok=True)

# Export both flight plans
plans = {
    'KBOS_KJFK': plan1,
    'KBOS_KORD':  plan2
}

all_plans = []
for route, plan in plans.items():
    plan['route'] = route
    all_plans.append(plan)

pd.DataFrame(all_plans).to_csv(f'{output_dir}\\flight_plans.csv', index=False)

# Export holding pattern entries
holding_results = []
test_cases = [
    (180, 45,  "Approaching from NE"),
    (180, 200, "Approaching from S"),
    (180, 300, "Approaching from NW"),
    (90,  135, "Approaching from SE"),
]
for inbound, heading, scenario in test_cases:
    entry, desc = holding_pattern_entry(inbound, heading)
    holding_results.append({
        'scenario':       scenario,
        'inbound_course': inbound,
        'aircraft_heading': heading,
        'entry_type':     entry,
        'procedure':      desc
    })

pd.DataFrame(holding_results).to_csv(
    f'{output_dir}\\holding_pattern_entries.csv', index=False
)

print(f"Exports saved to outputs/")
print(f"\n=== PilotGPT v2 Summary ===")
print(f"  E6B calculations:     Wind correction, TAS, density altitude, fuel")
print(f"  Holding patterns:     Direct, Teardrop, Parallel entry logic")
print(f"  Flight plan engine:   Full IFR plan with 45-min fuel reserve")
print(f"  AI agent:             5 tools, LangGraph ReAct, GPT-4o-mini")
print(f"  Airports in DB:       {len(airports)}")

Exports saved to outputs/

=== PilotGPT v2 Summary ===
  E6B calculations:     Wind correction, TAS, density altitude, fuel
  Holding patterns:     Direct, Teardrop, Parallel entry logic
  Flight plan engine:   Full IFR plan with 45-min fuel reserve
  AI agent:             5 tools, LangGraph ReAct, GPT-4o-mini
  Airports in DB:       10
